# 海凪の港町 — リアルな町の動画を GPU で描く

スマホでも動かせます。

1. 上のメニュー(スマホでは右上の **⋮**)→ **ランタイム** → **ランタイムのタイプを変更** → **T4 GPU** を選んで **保存**
2. **ランタイム** → **すべてのセルを実行**
3. 最後のセルに動画が出ます(15〜40分ほど)。動画を長押し、または **⋮** から保存できます

画質や長さは、下の「設定」のセルで変えられます。

In [ ]:
# 設定
SIZE = '1280x720'   # 画面の大きさ (1920x1080 にするともっと細かい、そのぶん時間がかかる)
FPS = 24            # 1秒あたりのコマ数
SECONDS = 12        # 動画の長さ(秒)
SAMPLES = 48        # 光の計算の回数(多いほどノイズが少ない)

In [ ]:
# GPU が使えるか確認
!nvidia-smi -L || echo 'GPU がありません: ランタイムのタイプを T4 GPU にしてください'

In [ ]:
# 町のデータを取ってくる(必要なフォルダだけ)
!rm -rf godzilla-quiz
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/apoptosis35-hash/godzilla-quiz.git
!cd godzilla-quiz && git sparse-checkout set uminagi-assets/tools uminagi-assets/scene uminagi-assets/textures
!ls godzilla-quiz/uminagi-assets/scene

In [ ]:
# Blender 5.2 (Linux 版) をダウンロード
import re, urllib.request, os
base = 'https://download.blender.org/release/'
url = None
for series in ('Blender5.2', 'Blender5.1'):
    try:
        idx = urllib.request.urlopen(base + series + '/').read().decode()
        files = sorted(set(re.findall(r'blender-5\.\d+\.\d+-linux-x64\.tar\.xz', idx)), key=lambda f: [int(n) for n in re.findall(r'\d+', f)[:3]])
        if files: url = base + series + '/' + files[-1]; break
    except Exception as e:
        print(series, e)
print(url)
!wget -q -O blender.tar.xz {url} && mkdir -p blender && tar xf blender.tar.xz -C blender --strip-components=1 && rm blender.tar.xz
!./blender/blender --version | head -1

In [ ]:
# 描画(GPU)。1コマごとに Saved: と出ます
!rm -rf godzilla-quiz/uminagi-assets/scene/film
!./blender/blender -b -P godzilla-quiz/uminagi-assets/tools/render_video.py -- --gpu --size {SIZE} --fps {FPS} --seconds {SECONDS} --samples {SAMPLES} 2>&1 | grep -E 'rendering on|no GPU|Saved|Error|Traceback'

In [ ]:
# 動画にして表示
!ffmpeg -y -loglevel error -framerate {FPS} -i godzilla-quiz/uminagi-assets/scene/film/%04d.png -c:v libx264 -crf 18 -preset slow -pix_fmt yuv420p -movflags +faststart uminagi_film.mp4
!ls -la uminagi_film.mp4
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('uminagi_film.mp4', 'rb').read()).decode()
HTML(f'<video controls playsinline style="width:100%" src="data:video/mp4;base64,{data}"></video>')

In [ ]:
# (PC のときは)ファイルとしてダウンロード
try:
    from google.colab import files
    files.download('uminagi_film.mp4')
except Exception as e:
    print(e)